In [0]:
%pip install mlxtend

In [0]:
%pip install xgboost

In [0]:
%restart_python

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mlflow
import mlflow.sklearn

sys.path.insert(0, os.path.abspath(".."))

from src.config import CFG
# import src.functions as fn
from src.functions import stratified_sample


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [0]:
gold_df_train = spark.read.table("workspace.fall_detection_project.gold_features_train").toPandas() # train_df

print(f"Train  : {gold_df_train.shape}")

non_feature_cols = ["label_binary", "label_multiclass", "split", "signal_id"]
feature_names = [c for c in gold_df_train.columns if c not in non_feature_cols]

X_train = gold_df_train.drop(columns=['label_binary', 'label_multiclass', 'split', 'signal_id'])
y_train_bin = gold_df_train['label_binary']
y_train_mul = gold_df_train['label_multiclass']

print(f"X_train : {X_train.shape}")
print(f"Features: {len(feature_names)}")

In [0]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

le = LabelEncoder()
y_train_encoded_bin = le.fit_transform(y_train_bin)

In [0]:

# models for shared, binary and multi class

shared_models = {
    "RandomForest"          : RandomForestClassifier(
        n_estimators=200, 
        max_depth=10, random_state=42, 
        class_weight="balanced"),
    
    "GradientBoosting"      : GradientBoostingClassifier(
        n_estimators=200, 
        learning_rate=0.1, max_depth=5, 
        random_state=42),
    
    "SVM"                   : SVC(
        kernel="rbf", 
        C=10, gamma="scale", 
        probability=True, class_weight="balanced"),
    
    "LogisticRegression"    : LogisticRegression(
        max_iter=1000, 
        class_weight="balanced", 
        random_state=42),
    
    "KNN"                   : KNeighborsClassifier(
        n_neighbors=5),
    
    "NeuralNetwork"         : MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        activation="relu",
        max_iter=500,
        random_state=42),
}


models_binary = {**shared_models,
    "XGBoost"               : XGBClassifier(
        n_estimators=200, 
        learning_rate=0.1, 
        max_depth=6, 
        random_state=42, 
        eval_metric="logloss", 
        scale_pos_weight=4.48), # handles class imbalance — ratio of ADL/Fall

}

models_multiclass = {**shared_models,

    "XGBoost"               : XGBClassifier(
        n_estimators=200, 
        learning_rate=0.1, 
        max_depth=6, 
        random_state=42, 
        eval_metric="logloss"),
    
}

In [0]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

scaled_models = ["SVM", "LogisticRegression", "NeuralNetwork"]

In [0]:
import time

from mlxtend.feature_selection import SequentialFeatureSelector as SFS

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_names)

for model_name, model in models_binary.items():
    if model_name != "RandomForest":
        continue
    
    start = time.time()
    sfs = SFS(
        model,
        k_features="best",
        forward=True,
        floating=True,
        scoring="f1",
        cv=5,
        n_jobs=-1,
        verbose=1
    )
    X_tr = X_train_scaled_df if model_name in scaled_models else X_train
    sfs.fit(X_tr, y_train_encoded_bin)
    elapsed = time.time() - start
    
    selected = list(sfs.k_feature_names_)
    print(f"{model_name} → {len(selected)} features  time={elapsed:.0f}s")
    print(f"Features: {selected}")

In [0]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_names)

for model_name, model in models_binary.items():
    sfs = SFS(
        model,
        k_features="best",
        forward=True,
        floating=True,
        scoring="f1",
        cv=5,
        n_jobs=-1,
        verbose=1
    )

    X_tr = X_train_scaled_df if model_name in scaled_models else X_train

    sfs.fit(X_tr, y_train_encoded_bin)
    selected = list(sfs.k_feature_names_)
    print(f"{model_name} → {len(selected)} features selected")
    print(f"Features: {selected}")

In [0]:
# Test on first 200 samples, features 0-20 only
sfs_test = SFS(
    RandomForestClassifier(n_estimators=50, random_state=42),
    k_features="best",
    forward=True,
    floating=True,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=2
)

X_train_np = X_train.values  # convert pandas DataFrame to numpy array

sfs_test.fit(X_train_np[:200, :20], y_train_encoded_bin[:200])

print(f"Selected features: {sfs_test.k_feature_idx_}")
print(f"Best score: {sfs_test.k_score_:.4f}")

In [0]:
print(f"Selected features: {sfs_test.k_feature_idx_}")
print(f"Best score: {sfs_test.k_score_:.4f}")

In [0]:
sub_df = stratified_sample(gold_df_train, "label_multiclass", 0.5)
# stratified_sample(gold_df_train, "label_multiclass")

In [0]:
gold_df_train.shape

In [0]:
sub_df.shape

In [0]:
# gold_df_train = spark.read.table("workspace.fall_detection_project.gold_features_train").toPandas() # train_df

gold_df_train_subset = stratified_sample(gold_df_train, "label_multiclass", 0.5)

print(f"Train  : {gold_df_train_subset.shape}")

non_feature_cols = ["label_binary", "label_multiclass", "split", "signal_id"]
feature_names = [c for c in gold_df_train_subset.columns if c not in non_feature_cols]

X_train_subset = gold_df_train_subset.drop(columns=['label_binary', 'label_multiclass', 'split', 'signal_id'])
y_train_subset_bin = gold_df_train_subset['label_binary']
y_train_subset_mul = gold_df_train_subset['label_multiclass']

print(f"X_train_subset : {X_train_subset.shape}")
print(f"Features: {len(feature_names)}")

In [0]:
scaler = StandardScaler()
X_train_subset_scaled = scaler.fit_transform(X_train_subset)

le = LabelEncoder()
y_train_subset_encoded_bin = le.fit_transform(y_train_subset_bin)

In [0]:
scaler = StandardScaler()
X_train_subset_scaled = scaler.fit_transform(X_train_subset)

scaled_models = ["SVM", "LogisticRegression", "NeuralNetwork"]

In [0]:
import time

from mlxtend.feature_selection import SequentialFeatureSelector as SFS

X_train_subset_scaled_df = pd.DataFrame(X_train_subset_scaled, columns=feature_names)

for model_name, model in models_binary.items():
    if model_name != "RandomForest":
        continue
    
    start = time.time()
    sfs = SFS(
        model,
        k_features="best",
        forward=True,
        floating=True,
        scoring="f1",
        cv=5,
        n_jobs=-1,
        verbose=1
    )
    X_tr = X_train_subset_scaled_df if model_name in scaled_models else X_train_subset
    sfs.fit(X_tr, y_train_subset_encoded_bin)
    elapsed = time.time() - start
    
    selected = list(sfs.k_feature_names_)
    print(f"{model_name} → {len(selected)} features  time={elapsed:.0f}s")
    print(f"Features: {selected}")

In [0]:
import time

from mlxtend.feature_selection import SequentialFeatureSelector as SFS

X_train_subset_scaled_df = pd.DataFrame(X_train_subset_scaled, columns=feature_names)

for model_name, model in models_binary.items():
    
    start = time.time()
    sfs = SFS(
        model,
        k_features="best",
        forward=True,
        floating=True,
        scoring="f1",
        cv=5,
        n_jobs=-1,
        verbose=1
    )
    X_tr = X_train_subset_scaled_df if model_name in scaled_models else X_train_subset
    sfs.fit(X_tr, y_train_subset_encoded_bin)
    elapsed = time.time() - start
    
    selected = list(sfs.k_feature_names_)
    print(f"{model_name} → {len(selected)} features  time={elapsed:.0f}s")
    print(f"Features: {selected}")